# Land Development Services — AI Support

This notebook demonstrates how to ask questions (English/Spanish) and log the answers into `session_log.json`.

**Quick start**
1. Run all cells top-to-bottom (Shift+Enter).
2. In the **Ask a question** cell (below), change the `question` string and re-run it.
3. When you are done, run the final cell to save the session log and view the report.


In [1]:
# Cell 2: Setup (imports + env)
from __future__ import annotations

import json
import os
import time
from pathlib import Path

from IPython.display import Markdown, display

import openai
from dotenv import load_dotenv

# Load environment variables from .env (make sure you have OPENAI_API_KEY set)
load_dotenv(override=False)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    display(Markdown('**Warning:** OPENAI_API_KEY is missing. AI responses will be mocked. Copy `.env.example` to `.env` and set your key.'))
else:
    openai.api_key = OPENAI_API_KEY

# Paths
ROOT = Path('.')
FAQ_PATH = ROOT / 'faq.md'
SESSION_LOG_PATH = ROOT / 'session_log.json'

# In-memory log for this session
session_log: list[dict] = []


In [5]:
# Cell 3: Load the FAQ and display it as a simple table
def load_faq(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    lines = path.read_text(encoding='utf-8').splitlines()

    entries: list[dict[str, str]] = []
    question = None
    answer_lines: list[str] = []
    for line in lines:
        stripped = line.strip()
        if stripped.startswith('**Q:') and stripped.endswith('**'):
            if question is not None:
                entries.append({'question': question, 'answer': ' '.join(answer_lines).strip()})
                answer_lines = []
            question = stripped[4:-2].strip()
        elif question is not None:
            answer_lines.append(line.rstrip())
    if question is not None:
        entries.append({'question': question, 'answer': ' '.join(answer_lines).strip()})
    return entries

faq_entries = load_faq(FAQ_PATH)

def display_faq_table(entries: list[dict[str, str]], max_rows: int = 20) -> None:
    if not entries:
        display(Markdown('**No FAQ entries found. Make sure `faq.md` exists.**'))
        return

    rows = entries[:max_rows]
    md_lines: list[str] = []
    md_lines.append('| Question | Answer |')
    md_lines.append('|---|---|')
    for e in rows:
        q = e['question'].replace('|', '\|')
        a = e['answer'].replace('|', '\|')
        md_lines.append(f'| {q} | {a} |')

    if len(entries) > max_rows:
        md_lines.append(f'\\n*Showing {max_rows} of {len(entries)} FAQ entries.*')

    display(Markdown('\\n'.join(md_lines)))

display_faq_table(faq_entries, max_rows=20)


<>:35: SyntaxWarning: invalid escape sequence '\|'
<>:36: SyntaxWarning: invalid escape sequence '\|'
<>:35: SyntaxWarning: invalid escape sequence '\|'
<>:36: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_787127/4252527483.py:35: SyntaxWarning: invalid escape sequence '\|'
  q = e['question'].replace('|', '\|')
/tmp/ipykernel_787127/4252527483.py:36: SyntaxWarning: invalid escape sequence '\|'
  a = e['answer'].replace('|', '\|')


| Question | Answer |\n|---|---|\n| What permits do I need to build a commercial property? | You need a Commercial Building Permit, a Zoning Compliance Certificate, and a Site Plan Approval. Submit all three to the Land Development Services office before breaking ground. |\n| How long does a commercial permit take to be approved? | Standard review takes 15–30 business days. Expedited review (additional fee) takes 5–10 business days. |\n| What documents are required for a commercial permit application? | - Completed permit application form - Site plan drawn to scale - Architectural and structural drawings - Proof of property ownership or authorization letter - Business license (if applicable) - Environmental impact assessment (for large projects) |\n| ¿Qué permisos necesito para construir una propiedad comercial? | Necesita un Permiso de Construcción Comercial, un Certificado de Cumplimiento de Zonificación y una Aprobación del Plan del Sitio. Preséntelos todos en la oficina de Servicios de Desarrollo de Tierras antes de comenzar la construcción.  ---  ## Residential Permits / Permisos Residenciales |\n| What permits do I need to build a new home? | You need a Residential Building Permit, an Electrical Permit, a Plumbing Permit, and a Mechanical Permit. All can be applied for simultaneously. |\n| How long does a residential permit take? | Typically 10–15 business days for standard single-family homes. |\n| Do I need a permit for a home addition or remodel? | Yes. Any structural change, addition, or major renovation requires a permit. Minor cosmetic work (painting, flooring) does not. |\n| What is the fee for a residential building permit? | Fees are based on the estimated construction value. Typically $0.50–$2.00 per square foot. Contact the office for an exact quote. |\n| ¿Necesito un permiso para remodelar mi casa? | Sí. Cualquier cambio estructural, adición o renovación importante requiere un permiso. Los trabajos cosméticos menores (pintura, pisos) no lo requieren.  ---  ## Plot / Land Verification / Verificación de Terreno |\n| How do I verify if my land is approved for permanent construction? | Submit a Land Use Verification Request with your parcel number. The department will confirm zoning classification, setback requirements, flood zone status, and utility easements. |\n| What is a zoning classification? | Zoning classifies land for specific uses: Residential (R1, R2), Commercial (C1, C2), Industrial (I1), or Mixed Use (MU). Construction must comply with the zone's allowed uses and building standards. |\n| What is a setback requirement? | A setback is the minimum distance a structure must be from property lines, roads, or other features. Violating setbacks can result in permit denial or required demolition. |\n| How do I find my parcel number? | Your parcel number (APN) appears on your property tax bill, deed, or the county assessor's website. |\n| What if my land is in a flood zone? | Construction in flood zones requires additional permits and must meet FEMA elevation standards. Flood zone status is determined during the Land Use Verification process. |\n| ¿Cómo verifico si mi terreno está aprobado para construcción permanente? | Presente una Solicitud de Verificación de Uso de Suelo con su número de parcela. El departamento confirmará la clasificación de zonificación, retiros requeridos, zona de inundación y servidumbres de servicios públicos. |\n| ¿Qué es la clasificación de zonificación? | La zonificación clasifica el terreno para usos específicos: Residencial (R1, R2), Comercial (C1, C2), Industrial (I1) o Uso Mixto (MU). La construcción debe cumplir con los usos permitidos y las normas de construcción de la zona.  ---  ## General / General |\n| What are the office hours? | Monday–Friday, 8:00 AM – 5:00 PM. Closed on federal holidays. |\n| Can I apply for permits online? | Yes. Visit the online permit portal on the department website. You can submit applications, upload documents, and track status online. |\n| How do I schedule an inspection? | Call the inspection hotline or request online at least 48 hours in advance. Inspections are available Monday–Friday. |\n| ¿Cuáles son los horarios de la oficina? | Lunes a viernes, 8:00 AM – 5:00 PM. Cerrado los días festivos federales. |\n\n*Showing 20 of 21 FAQ entries.*

In [7]:
# Cell 4: Helpers for asking questions and tracking sessions
def _detect_language(question: str) -> str:
    spanish_indicators = ['¿', '¡', ' cómo ', ' qué ', ' permiso', 'permiso', 'remodel', 'requisitos', 'zona', 'zonificación']
    lower = question.lower()
    for token in spanish_indicators:
        if token in lower:
            return 'spanish'
    return 'english'

def ask_land_support(question: str, *, return_raw: bool = False) -> dict:
    question = question.strip()
    if not question:
        raise ValueError('Question must not be empty.')

    if not OPENAI_API_KEY:
        # If no key is provided, return a warning message but still allow the notebook to run.
        mock_answer = (
            'OPENAI_API_KEY is missing. To get real AI responses, copy `.env.example` to `.env` and set your OPENAI_API_KEY.\n\n'
            'Meanwhile, use the FAQ table above as a reference for common questions.'
        )
        entry = {
            'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S%z'),
            'question': question,
            'language': 'unknown',
            'answer': mock_answer,
            'response_time_seconds': 0.0,
        }
        session_log.append(entry)
        return entry if return_raw else mock_answer

    language = _detect_language(question)
    system_prompt = (
        'You are an AI assistant for a municipal Land Development Services department. '
        'Answer the question in the same language that the user asked it (English or Spanish). '
        'Use the provided FAQ context when possible.'
    )

    start = time.time()
    response = openai.ChatCompletion.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question},
        ],
        temperature=0.2,
    )
    elapsed = time.time() - start

    answer = response.choices[0].message.content.strip()

    entry = {
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S%z'),
        'question': question,
        'language': language,
        'answer': answer,
        'response_time_seconds': round(elapsed, 3),
    }

    session_log.append(entry)

    if return_raw:
        return entry
    return answer


In [8]:
# Cell 5: Ask a question (edit this and rerun)
question = 'What permits do I need to build a new home?'  # <- Change this question and rerun this cell

answer = ask_land_support(question)
display(Markdown('**Question:** ' + question))
display(Markdown('**Answer:** ' + answer))


APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [9]:
# Cell 6: View the session log as a table
def display_session_table(entries: list[dict], max_rows: int = 20) -> None:
    if not entries:
        display(Markdown('**No session entries yet. Run the question cell above.**'))
        return

    rows = entries[-max_rows:]
    md_lines: list[str] = []
    md_lines.append('| # | Question | Answer | Time (s) |')
    md_lines.append('|---|---|---|---|')
    for idx, e in enumerate(rows, start=max(1, len(entries) - len(rows) + 1)):
        q = e['question'].replace('|', '\|')
        a = e['answer'].replace('|', '\|')
        t = e.get('response_time_seconds', '')
        md_lines.append(f'| {idx} | {q} | {a} | {t} |')

    display(Markdown('\\n'.join(md_lines)))

display_session_table(session_log, max_rows=20)


<>:12: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
<>:12: SyntaxWarning: invalid escape sequence '\|'
<>:13: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_787127/3800313947.py:12: SyntaxWarning: invalid escape sequence '\|'
  q = e['question'].replace('|', '\|')
/tmp/ipykernel_787127/3800313947.py:13: SyntaxWarning: invalid escape sequence '\|'
  a = e['answer'].replace('|', '\|')


**No session entries yet. Run the question cell above.**

In [10]:
# Cell 7: Save session log to disk and show a brief summary
SESSION_LOG_PATH.write_text(json.dumps(session_log, indent=2, ensure_ascii=False), encoding='utf-8')
display(Markdown(f'**Saved {len(session_log)} entries to `{SESSION_LOG_PATH}`**'))

# Optional: print a short summary
if session_log:
    times = [e.get('response_time_seconds', 0) for e in session_log]
    display(Markdown(f'*Average response time: {sum(times)/len(times):.3f} seconds*'))


**Saved 0 entries to `session_log.json`**